In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
train = pd.read_csv("../data/raw/fraudTrain.csv")
test = pd.read_csv("../data/raw/fraudTest.csv")

In [3]:
train["trans_date_trans_time"] = pd.to_datetime(train["trans_date_trans_time"])
test["trans_date_trans_time"] = pd.to_datetime(test["trans_date_trans_time"])

In [4]:
for data in [train, test]:
    data["hour"] = data["trans_date_trans_time"].dt.hour
    data["day_of_week"] = data["trans_date_trans_time"].dt.dayofweek
    data["month"] = data["trans_date_trans_time"].dt.month

In [5]:
for data in [train, test]:
    data["dob"] = pd.to_datetime(data["dob"])
    data["age"] = (
        data["trans_date_trans_time"].dt.year -
        data["dob"].dt.year
    )

In [6]:
merchant_frequency = train["merchant"].value_counts()

train["merchant_frequency"] = train["merchant"].map(merchant_frequency).fillna(0)
test["merchant_frequency"] = test["merchant"].map(merchant_frequency).fillna(0)

In [7]:
customer_frequency = train["cc_num"].value_counts()

train["customer_frequency"] = train["cc_num"].map(customer_frequency).fillna(0)
test["customer_frequency"] = test["cc_num"].map(customer_frequency).fillna(0)

In [8]:
category_frequency = train["category"].value_counts()

train["category_frequency"] = train["category"].map(category_frequency).fillna(0)
test["category_frequency"] = test["category"].map(category_frequency).fillna(0)

In [9]:
category_mapping = {
    category: index
    for index, category in enumerate(train["category"].unique())
}

train["category_encoded"] = train["category"].map(category_mapping).fillna(-1)
test["category_encoded"] = test["category"].map(category_mapping).fillna(-1)

In [10]:
features = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "unix_time",
    "hour",
    "day_of_week",
    "month",
    "age",
    "merchant_frequency",
    "customer_frequency",
    "category_frequency",
    "category_encoded"
]

X_train = train[features]
y_train = train["is_fraud"]

X_test = test[features]
y_test = test["is_fraud"]

In [11]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1296675, 15)
X_test: (555719, 15)
y_train: (1296675,)
y_test: (555719,)


In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
processed_train = pd.DataFrame(X_train_scaled, columns=features)
processed_test = pd.DataFrame(X_test_scaled, columns=features)

processed_train["is_fraud"] = y_train.values
processed_test["is_fraud"] = y_test.values

processed_train.to_csv("../data/processed/train_processed.csv", index=False)
processed_test.to_csv("../data/processed/test_processed.csv", index=False)

In [14]:
processed_train.head()

,amt,lat,long,city_pop,merch_lat,merch_long,unix_time,hour,day_of_week,month,age,merchant_frequency,customer_frequency,category_frequency,category_encoded,is_fraud
0,-0.407826,-0.484420,0.657620,-0.282589,-0.494354,0.593864,-1.858664,-1.878145,-0.941975,-1.504564,-0.864629,-1.472203,0.281849,-1.554721,-1.590209,0
1,0.230039,2.039120,-2.033870,-0.293670,2.078699,-2.030341,-1.858662,-1.878145,-0.941975,-1.504564,-0.289333,0.858371,1.631335,0.954273,-1.346599,0
2,0.934149,0.717754,-1.601537,-0.280406,0.902849,-1.592323,-1.858662,-1.878145,-0.941975,-1.504564,0.631140,-0.288060,-1.772008,-0.277296,-1.102989,0
3,-0.158132,1.515617,-1.590766,-0.287742,1.662886,-1.621848,-1.858660,-1.878145,-0.941975,-1.504564,0.343492,1.065784,-1.785476,1.287733,-0.859378,0
4,-0.177094,-0.023035,0.782279,-0.293835,0.026941,0.841909,-1.858651,-1.878145,-0.941975,-1.504564,-0.749570,-0.859390,0.267035,-0.874248,-0.615768,0
